In [1]:
exec(open('generate_dataset.py').read())

OSError: Cannot save file into a non-existent directory: '\home\claude\rides-analysis'

In [2]:
import os
print(os.getcwd())

C:\Users\anaga\rides-analysis


In [4]:
import os
os.chdir(r'C:\Users\anaga\rides-analysis')
exec(open('generate_dataset.py').read())

OSError: Cannot save file into a non-existent directory: '\home\claude\rides-analysis'

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

os.chdir(r'C:\Users\anaga\rides-analysis')

np.random.seed(42)
random.seed(42)

n = 5000

start_date = datetime(2024, 1, 1)
dates = [start_date + timedelta(days=random.randint(0, 364)) for _ in range(n)]

def random_hour():
    r = random.random()
    if r < 0.25: return random.randint(7, 9)
    elif r < 0.45: return random.randint(17, 20)
    elif r < 0.55: return random.randint(12, 13)
    else: return random.randint(0, 23)

hours = [random_hour() for _ in range(n)]
minutes = [random.randint(0, 59) for _ in range(n)]
timestamps = [d.replace(hour=h, minute=m) for d, h, m in zip(dates, hours, minutes)]

ride_types = np.random.choice(['Standard', 'Premium', 'Pool', 'XL'], n, p=[0.5, 0.2, 0.2, 0.1])
user_ages = np.random.normal(32, 10, n).clip(18, 65).astype(int)
user_genders = np.random.choice(['Male', 'Female', 'Other'], n, p=[0.52, 0.45, 0.03])

cities = ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai']
city = np.random.choice(cities, n, p=[0.3, 0.25, 0.2, 0.15, 0.1])

distance_km = np.random.exponential(8, n).clip(1, 40)

base_rates = {'Standard': 12, 'Premium': 20, 'Pool': 8, 'XL': 16}
fares = np.array([base_rates[rt] * dist * (1 + np.random.normal(0, 0.1))
                  for rt, dist in zip(ride_types, distance_km)]).clip(30, 800).round(2)

ratings = np.random.choice([1,2,3,4,5], n, p=[0.03, 0.05, 0.10, 0.32, 0.50])

is_peak = np.array([1 if (7 <= h <= 9) or (17 <= h <= 20) else 0 for h in hours])
surge = np.where(is_peak == 1, np.random.choice([1.0, 1.5, 2.0], n, p=[0.3, 0.4, 0.3]), 1.0)
final_fare = (fares * surge).round(2)

duration_min = (distance_km * 3.5 + np.random.normal(5, 3, n)).clip(5, 120).round(0).astype(int)

payment = np.random.choice(['UPI', 'Card', 'Cash', 'Wallet'], n, p=[0.45, 0.25, 0.15, 0.15])
cancelled = np.random.choice([0, 1], n, p=[0.92, 0.08])

def segment(age):
    if age < 25: return 'Student'
    elif age < 35: return 'Young Professional'
    elif age < 50: return 'Mid-Career'
    else: return 'Senior'

user_segment = [segment(a) for a in user_ages]
user_ids = np.random.choice(range(1, 2001), n)

df = pd.DataFrame({
    'ride_id': range(1, n+1),
    'user_id': user_ids,
    'timestamp': timestamps,
    'city': city,
    'ride_type': ride_types,
    'distance_km': distance_km.round(2),
    'duration_min': duration_min,
    'base_fare': fares,
    'surge_multiplier': surge,
    'final_fare': final_fare,
    'rating': ratings,
    'payment_method': payment,
    'cancelled': cancelled,
    'user_age': user_ages,
    'user_gender': user_genders,
    'user_segment': user_segment,
    'hour': hours,
    'day_of_week': [t.strftime('%A') for t in timestamps],
    'month': [t.strftime('%B') for t in timestamps],
    'is_peak_hour': is_peak,
})

df.to_csv('rides_data.csv', index=False)
print(f"Dataset created: {len(df)} rows")
print(df.head())

Dataset created: 5000 rows
   ride_id  user_id           timestamp       city ride_type  distance_km  \
0        1      432 2024-11-23 15:20:00    Chennai  Standard         3.55   
1        2      616 2024-02-27 02:04:00  Hyderabad        XL        26.87   
2        3      149 2024-01-13 10:45:00    Chennai      Pool         9.58   
3        4      440 2024-05-20 08:13:00     Mumbai   Premium         6.05   
4        5      219 2024-05-05 07:31:00     Mumbai  Standard        27.76   

   duration_min  base_fare  surge_multiplier  final_fare  rating  \
0            19      37.69               1.0       37.69       4   
1            93     388.27               1.0      388.27       5   
2            39      74.83               1.0       74.83       4   
3            21     134.13               1.5      201.20       5   
4           102     368.41               1.0      368.41       4   

  payment_method  cancelled  user_age user_gender        user_segment  hour  \
0           Cash      

In [9]:
import os
os.chdir(r'C:\Users\anaga\rides-analysis')
exec(open('analysis.py').read())

FileNotFoundError: [Errno 2] No such file or directory: '/home/claude/rides-analysis/rides_data.csv'

In [11]:
import os
os.chdir(r'C:\Users\anaga\rides-analysis')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0F1117',
    'axes.facecolor': '#1A1D2E',
    'axes.edgecolor': '#2E3250',
    'axes.labelcolor': '#C8D0F0',
    'xtick.color': '#8892C0',
    'ytick.color': '#8892C0',
    'text.color': '#C8D0F0',
    'grid.color': '#2E3250',
    'grid.linewidth': 0.6,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.titlecolor': '#E8EEFF',
})

ACCENT  = '#7C83FF'
ACCENT2 = '#FF6B9D'
ACCENT3 = '#00D4AA'
ACCENT4 = '#FFB347'
PALETTE = [ACCENT, ACCENT2, ACCENT3, ACCENT4, '#A78BFA']

df = pd.read_csv('rides_data.csv', parse_dates=['timestamp'])
df_active = df[df['cancelled'] == 0].copy()

print("=" * 55)
print("RIDES DATA ANALYSIS — SUMMARY")
print("=" * 55)
print(f"Total rides:        {len(df):,}")
print(f"Active rides:       {len(df_active):,}")
print(f"Cancellation rate:  {df['cancelled'].mean()*100:.1f}%")
print(f"Total revenue:      ₹{df_active['final_fare'].sum():,.0f}")
print(f"Avg fare:           ₹{df_active['final_fare'].mean():.2f}")
print(f"Avg rating:         {df_active['rating'].mean():.2f}")
print(f"Unique users:       {df['user_id'].nunique():,}")
print()

# ── Fig 1: Operations Dashboard ──────────────────────────────
fig = plt.figure(figsize=(18, 12), facecolor='#0F1117')
fig.suptitle('RIDES ANALYSIS — OPERATIONS DASHBOARD',
             fontsize=16, fontweight='bold', color='#E8EEFF', y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38,
                       left=0.06, right=0.97, top=0.93, bottom=0.07)

ax1 = fig.add_subplot(gs[0, :2])
hourly = df.groupby('hour').size()
bars = ax1.bar(hourly.index, hourly.values, color=ACCENT, alpha=0.85, width=0.75)
for i, b in enumerate(bars):
    h = b.get_x() + b.get_width()/2
    if (7 <= h <= 9) or (17 <= h <= 20):
        b.set_color(ACCENT2)
        b.set_alpha(1.0)
ax1.set_title('Ride Volume by Hour of Day')
ax1.set_xlabel('Hour')
ax1.set_ylabel('Number of Rides')
ax1.legend(fontsize=8, framealpha=0.2)
ax1.grid(axis='y', alpha=0.4)

ax2 = fig.add_subplot(gs[0, 2])
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df.groupby('day_of_week').size().reindex(dow_order)
ax2.barh(dow.index, dow.values, color=ACCENT3, alpha=0.85)
ax2.set_title('Rides by Day of Week')
ax2.set_xlabel('Rides')
ax2.grid(axis='x', alpha=0.4)

ax3 = fig.add_subplot(gs[1, :2])
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
monthly_rev = df_active.groupby('month')['final_fare'].sum().reindex(month_order)
ax3.plot(range(len(monthly_rev)), monthly_rev.values, color=ACCENT4, linewidth=2.5, marker='o', markersize=6)
ax3.fill_between(range(len(monthly_rev)), monthly_rev.values, alpha=0.15, color=ACCENT4)
ax3.set_xticks(range(len(monthly_rev)))
ax3.set_xticklabels([m[:3] for m in month_order], fontsize=8)
ax3.set_title('Monthly Revenue Trend (₹)')
ax3.set_ylabel('Revenue (₹)')
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
ax3.grid(alpha=0.4)

ax4 = fig.add_subplot(gs[1, 2])
city_rev = df_active.groupby('city')['final_fare'].sum().sort_values(ascending=True)
ax4.barh(city_rev.index, city_rev.values, color=PALETTE, alpha=0.9)
ax4.set_title('Revenue by City')
ax4.set_xlabel('Revenue (₹)')
ax4.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
ax4.grid(axis='x', alpha=0.4)

ax5 = fig.add_subplot(gs[2, 0])
rt_counts = df['ride_type'].value_counts()
wedges, texts, autotexts = ax5.pie(
    rt_counts.values, labels=rt_counts.index, autopct='%1.0f%%',
    colors=PALETTE, startangle=140,
    wedgeprops=dict(linewidth=1.5, edgecolor='#0F1117'))
for t in autotexts: t.set_color('#0F1117'); t.set_fontweight('bold'); t.set_fontsize(9)
ax5.set_title('Ride Type Mix')

ax6 = fig.add_subplot(gs[2, 1])
surge_data = df_active.groupby('is_peak_hour')['final_fare'].mean()
bars6 = ax6.bar(['Off-Peak', 'Peak Hour'], surge_data.values,
                color=[ACCENT, ACCENT2], alpha=0.9, width=0.5)
for bar, val in zip(bars6, surge_data.values):
    ax6.text(bar.get_x()+bar.get_width()/2, val+1, f'₹{val:.0f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold', color='#E8EEFF')
ax6.set_title('Avg Fare: Peak vs Off-Peak')
ax6.set_ylabel('Avg Fare (₹)')
ax6.grid(axis='y', alpha=0.4)

ax7 = fig.add_subplot(gs[2, 2])
rating_counts = df_active['rating'].value_counts().sort_index()
ax7.bar(rating_counts.index, rating_counts.values,
        color=[ACCENT2, ACCENT2, ACCENT4, ACCENT3, ACCENT3], alpha=0.9, width=0.7)
ax7.set_title('Rating Distribution')
ax7.set_xlabel('Star Rating')
ax7.set_ylabel('Count')
ax7.grid(axis='y', alpha=0.4)

plt.savefig('fig1_operations.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.close()
print("✓ Fig 1: Operations Dashboard saved")

# ── Fig 2: Demographics ───────────────────────────────────────
fig2, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor='#0F1117')
fig2.suptitle('RIDES ANALYSIS — USER DEMOGRAPHICS & BEHAVIOUR',
              fontsize=16, fontweight='bold', color='#E8EEFF', y=0.98)
for ax in axes.flat:
    ax.set_facecolor('#1A1D2E')

axes[0,0].hist(df['user_age'], bins=25, color=ACCENT, alpha=0.85, edgecolor='#0F1117')
axes[0,0].set_title('User Age Distribution')
axes[0,0].set_xlabel('Age')
axes[0,0].set_ylabel('Count')
axes[0,0].axvline(df['user_age'].mean(), color=ACCENT2, linestyle='--', linewidth=2, label=f"Mean: {df['user_age'].mean():.0f}")
axes[0,0].legend(fontsize=9)
axes[0,0].grid(alpha=0.4)

gender_rides = df.groupby('user_gender').agg(rides=('ride_id','count')).reset_index()
axes[0,1].bar(gender_rides['user_gender'], gender_rides['rides'], color=[ACCENT, ACCENT2, ACCENT3], alpha=0.9, width=0.5)
axes[0,1].set_title('Rides by Gender')
axes[0,1].set_ylabel('Number of Rides')
axes[0,1].grid(axis='y', alpha=0.4)

seg_rt = pd.crosstab(df['user_segment'], df['ride_type'], normalize='index') * 100
seg_order = ['Student', 'Young Professional', 'Mid-Career', 'Senior']
seg_rt = seg_rt.reindex(seg_order)
sns.heatmap(seg_rt, ax=axes[0,2], cmap='RdYlGn', annot=True, fmt='.0f',
            linewidths=0.5, linecolor='#0F1117', cbar_kws={'label': '%'})
axes[0,2].set_title('Ride Type Preference by Segment (%)')
axes[0,2].set_xlabel('')

pay_seg = df.groupby(['user_segment', 'payment_method']).size().unstack(fill_value=0)
pay_seg = pay_seg.reindex(seg_order)
pay_seg_pct = pay_seg.div(pay_seg.sum(axis=1), axis=0) * 100
pay_seg_pct.plot(kind='bar', ax=axes[1,0], color=PALETTE, alpha=0.9, width=0.7)
axes[1,0].set_title('Payment Method by Segment')
axes[1,0].set_xlabel('')
axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(), rotation=30, ha='right', fontsize=8)
axes[1,0].set_ylabel('% of Rides')
axes[1,0].legend(fontsize=8, framealpha=0.2)
axes[1,0].grid(axis='y', alpha=0.4)

df['age_group'] = pd.cut(df['user_age'], bins=[17,25,35,50,65], labels=['18-25','26-35','36-50','51-65'])
df_active = df[df['cancelled'] == 0].copy()
age_spend = df_active.groupby('age_group', observed=True)['final_fare'].mean()
bars_age = axes[1,1].bar(age_spend.index, age_spend.values, color=ACCENT4, alpha=0.85, width=0.5)
for bar, val in zip(bars_age, age_spend.values):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, val+0.5, f'₹{val:.0f}',
                   ha='center', va='bottom', fontsize=9, color='#E8EEFF')
axes[1,1].set_title('Average Spend by Age Group')
axes[1,1].set_ylabel('Avg Fare (₹)')
axes[1,1].grid(axis='y', alpha=0.4)

cancel_seg = df.groupby('user_segment')['cancelled'].mean() * 100
cancel_seg = cancel_seg.reindex(seg_order)
axes[1,2].bar(cancel_seg.index, cancel_seg.values, color=ACCENT2, alpha=0.85, width=0.5)
axes[1,2].set_title('Cancellation Rate by Segment (%)')
axes[1,2].set_xticklabels(axes[1,2].get_xticklabels(), rotation=20, ha='right', fontsize=8)
axes[1,2].set_ylabel('Cancellation Rate (%)')
axes[1,2].grid(axis='y', alpha=0.4)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('fig2_demographics.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.close()
print("✓ Fig 2: Demographics Dashboard saved")

# ── Fig 3: Clustering ─────────────────────────────────────────
user_features = df.groupby('user_id').agg(
    total_rides=('ride_id', 'count'),
    total_spend=('final_fare', 'sum'),
    avg_fare=('final_fare', 'mean'),
    avg_distance=('distance_km', 'mean'),
    avg_rating_given=('rating', 'mean'),
    pct_peak=('is_peak_hour', 'mean'),
    pct_premium=('ride_type', lambda x: (x == 'Premium').mean()),
    cancellation_rate=('cancelled', 'mean'),
).reset_index()

features = ['total_rides','total_spend','avg_fare','avg_distance',
            'avg_rating_given','pct_peak','pct_premium','cancellation_rate']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(user_features[features])

inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
user_features['cluster'] = km_final.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
user_features['pca1'] = X_pca[:, 0]
user_features['pca2'] = X_pca[:, 1]

cluster_stats = user_features.groupby('cluster')[features].mean()
cluster_names = {
    cluster_stats['total_spend'].idxmax(): 'High-Value Frequent',
    cluster_stats['pct_premium'].idxmax(): 'Premium Riders',
    cluster_stats['cancellation_rate'].idxmax(): 'Churners',
    cluster_stats['total_rides'].idxmin(): 'Occasional Riders',
}
used = set()
for k in list(cluster_names.keys()):
    if cluster_names[k] in used:
        cluster_names[k] = f'Segment {k}'
    used.add(cluster_names[k])
for c in [0,1,2,3]:
    if c not in cluster_names:
        cluster_names[c] = f'Segment {c}'
user_features['cluster_name'] = user_features['cluster'].map(cluster_names)

fig3, axes3 = plt.subplots(2, 2, figsize=(16, 12), facecolor='#0F1117')
fig3.suptitle('RIDES ANALYSIS — USER CLUSTERING (K-MEANS)',
              fontsize=16, fontweight='bold', color='#E8EEFF', y=0.98)
for ax in axes3.flat:
    ax.set_facecolor('#1A1D2E')

cluster_colors = {n: c for n, c in zip(user_features['cluster_name'].unique(), PALETTE)}

axes3[0,0].plot(K_range, inertias, color=ACCENT, linewidth=2.5, marker='o', markersize=7)
axes3[0,0].axvline(4, color=ACCENT2, linestyle='--', linewidth=1.5, label='Chosen k=4')
axes3[0,0].set_title('Elbow Method — Optimal k')
axes3[0,0].set_xlabel('Number of Clusters (k)')
axes3[0,0].set_ylabel('Inertia')
axes3[0,0].legend(fontsize=9)
axes3[0,0].grid(alpha=0.4)

for cname, grp in user_features.groupby('cluster_name'):
    axes3[0,1].scatter(grp['pca1'], grp['pca2'],
                       label=cname, alpha=0.55, s=25,
                       color=cluster_colors.get(cname, ACCENT))
axes3[0,1].set_title('User Clusters — PCA Projection')
axes3[0,1].set_xlabel(f'PCA 1 ({pca.explained_variance_ratio_[0]*100:.0f}% var)')
axes3[0,1].set_ylabel(f'PCA 2 ({pca.explained_variance_ratio_[1]*100:.0f}% var)')
axes3[0,1].legend(fontsize=8, framealpha=0.2)
axes3[0,1].grid(alpha=0.3)

cluster_counts = user_features['cluster_name'].value_counts()
axes3[1,0].bar(cluster_counts.index, cluster_counts.values,
               color=[cluster_colors.get(n, ACCENT) for n in cluster_counts.index], alpha=0.9, width=0.6)
axes3[1,0].set_title('Users per Cluster')
axes3[1,0].set_ylabel('Number of Users')
axes3[1,0].set_xticklabels(axes3[1,0].get_xticklabels(), rotation=20, ha='right', fontsize=9)
axes3[1,0].grid(axis='y', alpha=0.4)

radar_features = ['total_rides','avg_fare','pct_peak','pct_premium','cancellation_rate']
cluster_radar = user_features.groupby('cluster_name')[radar_features].mean()
cluster_radar_norm = (cluster_radar - cluster_radar.min()) / (cluster_radar.max() - cluster_radar.min())
x = np.arange(len(radar_features))
width = 0.2
for i, (cname, row) in enumerate(cluster_radar_norm.iterrows()):
    axes3[1,1].bar(x + i*width, row.values, width=width,
                   label=cname, color=cluster_colors.get(cname, ACCENT), alpha=0.88)
axes3[1,1].set_xticks(x + width*1.5)
axes3[1,1].set_xticklabels(['Rides', 'Avg Fare', 'Peak%', 'Premium%', 'Cancel%'], fontsize=9)
axes3[1,1].set_title('Cluster Profile Comparison (Normalised)')
axes3[1,1].set_ylabel('Normalised Score')
axes3[1,1].legend(fontsize=8, framealpha=0.2)
axes3[1,1].grid(axis='y', alpha=0.4)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('fig3_clustering.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.close()
print("✓ Fig 3: Clustering saved")

print("\n" + "=" * 55)
print("ALL CHARTS SAVED SUCCESSFULLY!")
print("=" * 55)

RIDES DATA ANALYSIS — SUMMARY
Total rides:        5,000
Active rides:       4,621
Cancellation rate:  7.6%
Total revenue:      ₹660,770
Avg fare:           ₹142.99
Avg rating:         4.22
Unique users:       1,842

✓ Fig 1: Operations Dashboard saved
✓ Fig 2: Demographics Dashboard saved
✓ Fig 3: Clustering saved

ALL CHARTS SAVED SUCCESSFULLY!
